In [1]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from numpy import pi as π, sin,cos

In [2]:
# ==========================================
# 1. PARAMETERS & SETUP
# ==========================================
N = 300  # Grid resolution (NxN)
alpha = 0*π / 8  # Example value for alpha

# Example probabilities (Replace with your actual 'Pt' values)

P_ent = 1.0

if P_ent == 1.0:
    P_HH = P_HV = P_VH = P_VV = 0.0
else:
    P_HH = 0.0030
    P_HV = 0.0138
    P_VH = 0.0278
    P_VV = 0.0447

# DEFINING THE MAPπNG: You must adjust this to your specific framework
def get_phis(alpha, beta):
    """Map alpha and beta to the phi angles used in the CR equation."""
    phi0 = 0
    phi1 = alpha/2
    phi2 = alpha/2+beta
    phi3 = 3*π/8
    return phi0, phi1, phi2, phi3

# Shorthand trigonometric functions
def C(x): return np.cos(x)
def S(x): return np.sin(x)

# ==========================================
# 2. EQUATIONS
# ==========================================
def calc_P_corr(theta, alpha, beta):
    """Calculates P_Corr^T (Mixed)"""
    term1 = P_ent * 0.5 * (
        C(alpha)**4 + S(alpha)**4 + 2 * S(alpha)**2 * C(alpha)**2 * C(theta) +
        C(alpha+beta)**4 + S(alpha+beta)**4 + 2 * S(alpha+beta)**2 * C(alpha+beta)**2 * C(theta)
    )
    term2 = ((P_HH + P_VV) / 2) * (C(alpha)**4 + S(alpha)**4 + C(alpha+beta)**4 + S(alpha+beta)**4)
    term3 = (P_HV + P_VH) * (C(alpha)**2 * S(alpha)**2 + C(alpha+beta)**2 * S(alpha+beta)**2)
    return term1 + term2 + term3

def calc_CR(theta, alpha, beta):
    """Calculates CR^M"""
    phi0, phi1, phi2, phi3 = get_phis(alpha, beta)
    
    term1 = P_ent * (
        C(theta) * (S(2*phi0)*(S(2*phi1) - S(2*phi3)) + S(2*phi2)*(S(2*phi3) + S(2*phi1))) +
        C(2*phi0)*(C(2*phi1) - C(2*phi3)) + C(2*phi2)*(C(2*phi3) + C(2*phi1))
    )
    term2 = (P_HH + P_VV - P_HV - P_VH) * (
        C(2*phi0)*(C(2*phi1) - C(2*phi3)) + C(2*phi2)*(C(2*phi3) + C(2*phi1))
    )
    return term1 + term2

# ==========================================
# 3. GRID GENERATION
# ==========================================
# x-axis: beta from 0 to π
# y-axis: theta from 0 to 2*π
beta_vals = np.linspace(0, π, N)
theta_vals = np.linspace(0, 2 * π, N)
beta_grid, theta_grid = np.meshgrid(beta_vals, theta_vals)

P_corr_grid = calc_P_corr(theta_grid, alpha, beta_grid)
CR_grid = calc_CR(theta_grid, alpha, beta_grid)

# ==========================================
# 4. REGION CLASSIFICATION
# ==========================================
# Find xi: minimum P_corr where CR >= 2
cr_mask = CR_grid >= 2

if np.any(cr_mask):
    xi = np.min(P_corr_grid[cr_mask])
    print(f"Calculated xi limit: {xi:.4f}")
else:
    xi = 0
    print("Warning: No region satisfies CR >= 2 with the current parameters.")

# Initialize the discrete regions grid
Z_regions = np.zeros_like(P_corr_grid)

min_limit = min(xi, 0.89)

# Region 3.3: Opposite inequality of P_corr for the minimum between xi and 0.89
Z_regions[P_corr_grid < min_limit] = 1 

# Region 3.2: P_corr >= xi such that CR >= 2
Z_regions[cr_mask] = 2 

# Region 3.1: P_corr >= 0.89
# (This overrides 3.2 if they overlap, keeπng the strict 0.89 boundary distinct)
Z_regions[P_corr_grid >= 0.89] = 3 

Calculated xi limit: 0.5000


In [11]:
# 1. Variables for formatting quantities outside figure
tick_font_size = 20
legend_font_size = 20
axes_label_font_size = 30
title_fontsize = 30

font_family = "Arial"
fig = go.Figure(data=go.Contour(
    z=Z_regions,
    x=beta_vals,
    y=theta_vals,
    colorscale=[
        [0.0, '#787878'],  # dead zone
        [0.5, '#086375'],  # bell test possible
        [1.0, '#ed0707']   # over Shannon limit
    ],
    contours=dict(
        start=1,
        end=3,
        size=1,
        coloring='heatmap'
    ),
    colorbar=dict(
        title="<b>Regions</b>",
        tickfont=dict(size=colorbar_tick_fontsize, family=font_family),
        tickmode='array',
        tickvals=[1.33, 2, 2.66], # Centered ticks for a 3-color scale
        ticktext=[
            'Dead Zone',            # Replaces generic Region 3.3 text
            'Bell Test Security',   # Replaces generic Region 3.2 text
            'Over Shannon limit'    # Replaces generic Region 3.1 text
        ]
    )
))

fig.update_layout(
    yaxis=dict(
        title=dict(
            text='θ',
            font=dict(size=axes_label_font_size, family=font_family)
        ), 
        tickfont=dict(size=tick_font_size, family=font_family),                    
        tickmode='array',
        tickvals=[0, np.pi/3, 2*np.pi/3, np.pi, 4*np.pi/3, 5*np.pi/3, 2*np.pi],
                ticktext=['0', 'π/3', '2π/3', 'π', '4π/3', '5π/3', '2π'],
        range=[0, 2*π],
        showgrid=True, gridcolor='lightgray',zeroline=True,
        showline=True, linewidth=1, linecolor='gray', mirror=True
    ),
    xaxis=dict(
        title=dict(
            text='β)',  # Update y-axis label as needed
            font=dict(size=axes_label_font_size, family=font_family)
        ),
        tickfont=dict(size=tick_font_size, family=font_family),
        tickmode='array',
        tickvals=[0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi],
                ticktext=['0', 'π/4', 'π/2', '3π/4', 'π'],
        showgrid=True, gridcolor='lightgray',
        showline=True, linewidth=1, linecolor='gray', mirror=True,
    ),
    title=dict(
        text=f"(a)\t\tα = π/8",
        font=dict(size=title_fontsize, family=font_family),
        x=0.1, 
        y=0.96
    ),
    autosize=False,
    width=750,
    height=750,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=14)
)

fig.show()
fig.write()

NameError: name 'colorbar_tick_fontsize' is not defined